# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%%configure
{
    "--extra-jars": "s3://path/to/postgresql-42.7.3.jar"
}

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 3
%connections postgresql_connection

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import regexp_extract, regexp_replace, col
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

In [ ]:
sc._jsc.sc().removeJar("s3://path/to/postgresql-42.7.3.jar")

In [ ]:
DB_CONNECTION_URL = 'jdbc:postgresql://originator-data-vault.cmyschvxh2v8.us-west-1.rds.amazonaws.com:5432/postgres'
DB_HOST = 'originator-data-vault.cmyschvxh2v8.us-west-1.rds.amazonaws.com'
DB_PASSWORD = 'XKZ6ruq3wtd4mcw!qwb'
DB_USERNAME = 'delos'

In [ ]:
prepayment_query = """SELECT
        loan_number as loan_number,
        effective_date,
        principal_prepayment_amount,
        scheduled_principal_amount,
        interest_amount,
        mob as mob_data_model
     FROM loan_transaction
"""
# query frame with prepayment_query for building blocks
prepayment_df = spark.read \
    .format("jdbc") \
    .option("url", DB_CONNECTION_URL) \
    .option("driver", "org.postgresql.Driver") \
    .option("query", prepayment_query) \
    .option("user", DB_USERNAME) \
    .option("password", DB_PASSWORD) \
    .load()
prepayment_df.filter(prepayment_df.loan_number == 18739164).show(5)

In [ ]:
char_score_df = spark.createDataFrame(
    [
        (786,'s004s','[0.0,12.0)',34.0),
        (787,'s004s','[12.0,24.0)',55.0),
        (788,'s004s','[24.0,36.0)',57.0),
        (789,'s004s','[36.0,48.0)',61.0),
        (790,'s004s','[48.0,60.0)',63.0),
        (791,'s004s','[60.0,72.0)',64.0),
        (792,'s004s','[72.0,84.0)',69.0),
        (793,'s004s','[84.0,96.0)',70.0),
        (794,'s004s','[96.0,108.0)',70.0),
        (795,'s004s','[108.0,120.0)',70.0),
        (796,'s004s','[120.0,132.0)',70.0),
        (797,'s004s','[132.0,144.0)',70.0),
        (798,'s004s','[144.0,156.0)',70.0),
        (799,'s004s','[156.0,180.0)',70.0),
        (800,'s004s','[180.0,)',70.0),
        (801,'at36s','(,0.0)',63.0),
        (802, 'at36s', '[0.0,1.0)',58.0),
        (803, 'at36s', '[1.0,2.0)',59.0),
        (804, 'at36s', '[2.0,3.0)',59.0),
        (805, 'at36s', '[3.0,6.0)',59.0),
        (806, 'at36s', '[6.0,9.0)',59.0),
        (807, 'at36s', '[9.0,12.0)',59.0),
        (808, 'at36s', '[12.0,18.0)',60.0),
        (809, 'at36s', '[18.0,24.0)',61.0),
        (810, 'at36s', '[24.0,36.0)',64.0),
        (811, 'at36s', '[36.0,48.0)',67.0),
        (812, 'at36s', '[48.0,60.0)',70.0),
        (813, 'at36s', '[60.0,72.0)',72.0),
        (814, 'at36s', '[72.0,999.0)',72.0),
        (815, 'at36s', '[999.0,1000.0)',73.0),
        (816, 'at36s', '[1000.0,)',63.0),
        (1041, 'at36s', 'empty',63.0),
        (1034,'s004s','empty',62.0),
    ],
    ["id","characteristic","value_range","partial_score"]
)

input_data_df = spark.createDataFrame(
    [
        (1,13.5,2.7),
        (2,74.0,39.0)
    ],
    ["loan_id","s004s","at36s"]
)

char_score_df = char_score_df.withColumn("lower_bound", regexp_extract("value_range", r"[\[\(](\d+\.?\d*)", 1).cast("double")) \
       .withColumn("upper_bound", regexp_extract("value_range", r"(\d+\.?\d*)\s*[\]\)]", 1).cast("double"))

df = input_data_df
characteristics = ["s004s", "at36s"]

# Loop through each characteristic and join
for char in characteristics:
    df = df.alias("input").join(
        char_score_df.alias("char"),
        (col(f"input.{char}") >= col("char.lower_bound")) & 
        (col(f"input.{char}") < col("char.upper_bound")) & 
        (col("char.characteristic") == char),
        "left"
    ).select(
        [col(f"input.{c}") for c in df.columns] +  # Keep all previous columns
        [col("char.partial_score").alias(f"{char}_score")]  # Add new score column
    )
    
# Add total_score column
score_columns = [f"{char}_score" for char in characteristics]
df = df.withColumn("total_score", sum(col(c) for c in score_columns))

df.show()
result_df.show(100)

In [ ]:
loan_product_df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://originator-data-vault.cmyschvxh2v8.us-west-1.rds.amazonaws.com:5432/postgres") \
    .option("dbtable", "characteristic_scores") \
    .option("user", "delos") \
    .option("password", "XKZ6ruq3wtd4mcw!qwb") \
    .load() \
    .select("id", "characteristic")